<a href="https://colab.research.google.com/github/mf1060/GenAI/blob/main/HW1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Michael Furey

Dr. Forouraghi

CSC 688

1/27/2026


This notebook describes the results for Assignment 1: The GPT-2 Stress Test. The goal of this notebook is to observe responses with varying temperatures to a GPT-2 model. This notebook will use 20 tokens to test the model.

This notebook uses code for creating a response with GPT-2 and distilgpt2 models from this [Github page.](https://github.com/bforoura/GENAI26/blob/main/Module1/LLM_Text_Generator.ipynb) This includes comments from this page documenting this code.

# Using GPT - 2

In [1]:
# Install & import the needed libraries
# The following installs dependencies for the model.

!pip install -q transformers torch

!pip install triton torchao



In [2]:
import torch
import torch.nn.functional as F
from transformers import GPT2Tokenizer, GPT2LMHeadModel
import os
os.environ["TQDM_DISABLE"] = "1" # Disables progress bar widgets error caused by GPT


In [3]:
# The following is the prompt used to generate the response.

# I would like to learn about genAI, and I also like code. What should I do?

text = "I would like to learn about genAI, and I also like code. What should I do?"


In [4]:
# The tokenization step typically creates subword tokens, and not necessarily whole words

# Creating a tokenizer for gpt2
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
# Creating a model for gpt2
model = GPT2LMHeadModel.from_pretrained("gpt2")
model.eval()

#Creates tokens from the text above with the tokenizer
tokens = tokenizer.encode(text, return_tensors="pt")

#print("Token IDs:", tokens.tolist()[0])
#print("Tokens:")
for tid in tokens[0]:
    print(f"{tid.item():>6} → '{tokenizer.decode(tid)}'")

# The embeddings
with torch.no_grad():
    # Token embeddings
    token_embeds = model.transformer.wte(tokens)

    # Positional embeddings
    positions = torch.arange(tokens.size(1)).unsqueeze(0)
    pos_embeds = model.transformer.wpe(positions)

    embeddings = token_embeds + pos_embeds

# (batch_size, sequence_length, embedding_dim)
#print("Embedding shape:", embeddings.shape)

# The transformer forward pass ensures that each token now contains contextual information from previous tokens.
# This is the most important step conceptually, because this is where the model goes from isolated words to understanding a sentence.

with torch.no_grad():

    # Send the embedding vectors through all transformer layers (for GPT-2, it is 12 layers)
    outputs = model.transformer(inputs_embeds=embeddings)

    # Each layer, applies the self-attention mechanism and goes through a feed-forward NN
    hidden_states = outputs.last_hidden_state

#print("Hidden state shape:", hidden_states.shape)

# Logits for the next token. This gives one score per vocabulary token (~50k tokens)

with torch.no_grad():
    last_hidden = hidden_states[:, -1, :]
    logits = model.lm_head(last_hidden)

#print("Logits shape:", logits.shape)

# Softmax → probabilities: this is the actual probability distribution the model uses

probs = F.softmax(logits, dim=-1)

top_probs, top_ids = torch.topk(probs, k=10)

#print("Top 10 next-token probabilities:")
for p, tid in zip(top_probs[0], top_ids[0]):
    token = tokenizer.decode(tid)
    #print(f"{token!r:>12} : {p.item():.4f}")

def sample_next_token(logits, temperature=1.2, top_k=50):
    logits = logits / temperature

    if top_k is not None:
        values, indices = torch.topk(logits, top_k)
        probs = F.softmax(values, dim=-1)
        choice = torch.multinomial(probs, 1)
        return indices[0, choice]
    else:
        probs = F.softmax(logits, dim=-1)
        return torch.multinomial(probs, 1)

#next_token_id = sample_next_token(logits, temperature=0.8, top_k=40)
#print("Sampled token:", tokenizer.decode(next_token_id[0]))

# Full loop (generate multiple tokens)
#Modifies the function so that temperature and top_k are in the method signature.
def generate_step_by_step(prompt, steps=10, temp_var=0.8, top_k_var=40):
    tokens = tokenizer.encode(prompt, return_tensors="pt")

    for _ in range(steps):
        with torch.no_grad():
            outputs = model(tokens)
            logits = outputs.logits[:, -1, :]
            next_token = sample_next_token(logits, temperature=temp_var, top_k=top_k_var)

        tokens = torch.cat([tokens, next_token], dim=1)
        print(tokenizer.decode(tokens[0]))

    #Returns the final response instead of printing it
    return tokenizer.decode(tokens[0])




#next_token_id = sample_next_token(logits, temperature=0.8, top_k=40)
#print("Sampled token:", tokenizer.decode(next_token_id[0]))



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


    40 → 'I'
   561 → ' would'
   588 → ' like'
   284 → ' to'
  2193 → ' learn'
   546 → ' about'
  2429 → ' gen'
 20185 → 'AI'
    11 → ','
   290 → ' and'
   314 → ' I'
   635 → ' also'
   588 → ' like'
  2438 → ' code'
    13 → '.'
  1867 → ' What'
   815 → ' should'
   314 → ' I'
   466 → ' do'
    30 → '?'


## Running the prompt with different temperatures


In [5]:
#Running the GPT2 prompt at 0.1 temperature and top-k=40
test_01_gpt2 = generate_step_by_step(text, steps=20, temp_var=0.1, top_k_var = 40)
print("Resulting Prompt: " + test_01_gpt2)


I would like to learn about genAI, and I also like code. What should I do?

I would like to learn about genAI, and I also like code. What should I do?


I would like to learn about genAI, and I also like code. What should I do?

I
I would like to learn about genAI, and I also like code. What should I do?

I'm
I would like to learn about genAI, and I also like code. What should I do?

I'm not
I would like to learn about genAI, and I also like code. What should I do?

I'm not sure
I would like to learn about genAI, and I also like code. What should I do?

I'm not sure what
I would like to learn about genAI, and I also like code. What should I do?

I'm not sure what to
I would like to learn about genAI, and I also like code. What should I do?

I'm not sure what to do
I would like to learn about genAI, and I also like code. What should I do?

I'm not sure what to do with
I would like to learn about genAI, and I also like code. What should I do?

I'm not sure what to do with this
I would li

When temperature = 0.1, the model repeated a non-answer: "I'm not sure what to do with this."

In [6]:
#Running the GPT2 prompt at 0.8 temperature and top-k=40
test_08_gpt2 = generate_step_by_step(text, steps=20, temp_var=0.8, top_k_var = 40)
print("Resulting Prompt: " + test_08_gpt2)

I would like to learn about genAI, and I also like code. What should I do?

I would like to learn about genAI, and I also like code. What should I do?


I would like to learn about genAI, and I also like code. What should I do?

The
I would like to learn about genAI, and I also like code. What should I do?

The first
I would like to learn about genAI, and I also like code. What should I do?

The first thing
I would like to learn about genAI, and I also like code. What should I do?

The first thing I
I would like to learn about genAI, and I also like code. What should I do?

The first thing I would
I would like to learn about genAI, and I also like code. What should I do?

The first thing I would like
I would like to learn about genAI, and I also like code. What should I do?

The first thing I would like to
I would like to learn about genAI, and I also like code. What should I do?

The first thing I would like to mention
I would like to learn about genAI, and I also like code. What shou

When temperature = 0.8, the model produced a response that appeared to be the beginning of an answer to the question.  

In [7]:
#Running the GPT2 prompt at 2.0 temperature and top-k=40
test_2_gpt2 = generate_step_by_step(text, steps=20, temp_var=2.0, top_k_var = 40)
print("Resulting Prompt: " + test_2_gpt2)

I would like to learn about genAI, and I also like code. What should I do? A
I would like to learn about genAI, and I also like code. What should I do? A lot
I would like to learn about genAI, and I also like code. What should I do? A lot (
I would like to learn about genAI, and I also like code. What should I do? A lot (if
I would like to learn about genAI, and I also like code. What should I do? A lot (if you
I would like to learn about genAI, and I also like code. What should I do? A lot (if you include
I would like to learn about genAI, and I also like code. What should I do? A lot (if you include me
I would like to learn about genAI, and I also like code. What should I do? A lot (if you include me):
I would like to learn about genAI, and I also like code. What should I do? A lot (if you include me): if
I would like to learn about genAI, and I also like code. What should I do? A lot (if you include me): if I
I would like to learn about genAI, and I also like code. What should I do?

When temperature = 2.0, the model produced a response to the question like it was a journal entry.

# GPT - 2: Conclusions

I used the following [tutorial](https://www.geeksforgeeks.org/html/markdown-tables/) for making markdown tables as well as this [reference sheet](https://www.markdownlang.com/cheatsheet/).

The responses are graded with the following rubric:

| Description | Score Range |
|---| --- |
|(Perfect) Flawless grammar and logical flow.| 9-10|
|(High): Stays on topic; perhaps one slightly odd word choice. | 7–8 |
|(Fair): You can follow the "vibe," but there are logical leaps or awkward phrasing. | 5–6 |
|(Poor): Fragments of meaning exist, but the sentences don't connect. | 3–4
| (Incoherent): Total gibberish, random symbols, or word salad. | 1–2



The following are the results for the experiment above including my scores.


| Trial | Temperature (T) |	Predicted Behavior |	Model Response |	Model Coherence (1-10) | |
|---| --- | --- | --- | --- | ---|
|A	| 0.1 |	Conservative| I'm not sure what to do with this. I'm not sure what to do with | 7
|B	|0.8	|Creative	| The first thing I would like to mention is that I am a programmer. I am not |8
|C	|2.0	|Chaos |A lot (if you include me): if I like the product much and will not start it too|	3

**Did your model repeat any words or phrases?**

The only run that repeated words was the conservative (0.1 temperature) run. In that run, the model repeated "I'm not sure what to do with this." Because lower temperature models sample only the most likely tokens, we may expect the sentences to repeat themselves or follow a loop. This is consistent behavior for what we would expect from a conservative model.

The creative and chaos runs did not tend to repeat words or phrases except they both used the first person. Because these models are higher temperature, we would expect more variation in the tokens sampled.

**Did the model use real words, or did it start outputting random characters and punctuation? Explain how the "Probability Distribution" changed to allow this.**

All models used real words. The only run that produced odd punctuation was the chaos run which produced a colon and unnecessary parentheses. I believe most models produced a response with real words, despite varying temperature, because of the top_k setting. I would imagine that a high temperature would level out the probability distribution for many tokens (even random characters or punctuation) to have as much likelihood of being sampled as a more relevant token. However, the top_k setting prevents some of the lowest probability tokens from being chosen.

**If you were building a medical AI to give prescriptions or advice, which temperature would you use?**

For GPT-2, I would probably use a lower temperature to provide advice, because I would not want lower probability tokens to be sampled in the model. However, I may choose a higher temperature setting than 0.1, because the model may tend to repeat itself. I may try to use a temperature just short of a creative response (temperature = 0.5-0.7) to create responses consistent with medical advice.

**If you were building an AI to write a surrealist dream-journal, which would you use?**

I would likely set temperature to 2.0. This temperature appears consistent with surrealism, because the chaos run created a surrealist response. The chaos run created a narrative instead of a response to the question. If I would like to adopt a model for a surrealist dream journal, I would use a higher temperature setting like 2.0. Even a 0.8 temperature setting may produce a less creative response and higher temperature models would create a sense that the model was free-associating.

# Extra Credit

Extra Credit

This run uses a distilled version of GPT2 to compare results to the original. The following uses the same prompt, but the distilgpt2 model should use fewer layers.

In [8]:
# Enter your own input text

# I would like to learn about genAI, and I also like code. What should I do?

text = "I would like to learn about genAI, and I also like code. What should I do?"


In [9]:
# The tokenization step typically creates subword tokens, and not necessarily whole words

tokenizer = GPT2Tokenizer.from_pretrained("distilgpt2")
model = GPT2LMHeadModel.from_pretrained("distilgpt2")
model.eval()

tokens = tokenizer.encode(text, return_tensors="pt")

#print("Token IDs:", tokens.tolist()[0])
#print("Tokens:")
for tid in tokens[0]:
    print(f"{tid.item():>6} → '{tokenizer.decode(tid)}'")

# The embeddings
with torch.no_grad():
    # Token embeddings
    token_embeds = model.transformer.wte(tokens)

    # Positional embeddings
    positions = torch.arange(tokens.size(1)).unsqueeze(0)
    pos_embeds = model.transformer.wpe(positions)

    embeddings = token_embeds + pos_embeds

# (batch_size, sequence_length, embedding_dim)
#print("Embedding shape:", embeddings.shape)

# The transformer forward pass ensures that each token now contains contextual information from previous tokens.
# This is the most important step conceptually, because this is where the model goes from isolated words to understanding a sentence.

with torch.no_grad():

    # Send the embedding vectors through all transformer layers (for GPT-2, it is 12 layers)
    outputs = model.transformer(inputs_embeds=embeddings)

    # Each layer, applies the self-attention mechanism and goes through a feed-forward NN
    hidden_states = outputs.last_hidden_state

#print("Hidden state shape:", hidden_states.shape)

# Logits for the next token. This gives one score per vocabulary token (~50k tokens)

with torch.no_grad():
    last_hidden = hidden_states[:, -1, :]
    logits = model.lm_head(last_hidden)

#print("Logits shape:", logits.shape)

# Softmax → probabilities: this is the actual probability distribution the model uses

probs = F.softmax(logits, dim=-1)

top_probs, top_ids = torch.topk(probs, k=10)

#print("Top 10 next-token probabilities:")
for p, tid in zip(top_probs[0], top_ids[0]):
    token = tokenizer.decode(tid)
    #print(f"{token!r:>12} : {p.item():.4f}")

def sample_next_token(logits, temperature=1.2, top_k=50):
    logits = logits / temperature

    if top_k is not None:
        values, indices = torch.topk(logits, top_k)
        probs = F.softmax(values, dim=-1)
        choice = torch.multinomial(probs, 1)
        return indices[0, choice]
    else:
        probs = F.softmax(logits, dim=-1)
        return torch.multinomial(probs, 1)

#next_token_id = sample_next_token(logits, temperature=0.8, top_k=40)
#print("Sampled token:", tokenizer.decode(next_token_id[0]))

# Full loop (generate multiple tokens)
#Modifies the function so that temperature and top_k are in the method signature.
def generate_step_by_step(prompt, steps=10, temp_var=0.8, top_k_var=40):
    tokens = tokenizer.encode(prompt, return_tensors="pt")

    for _ in range(steps):
        with torch.no_grad():
            outputs = model(tokens)
            logits = outputs.logits[:, -1, :]
            next_token = sample_next_token(logits, temperature=temp_var, top_k=top_k_var)

        tokens = torch.cat([tokens, next_token], dim=1)
        print(tokenizer.decode(tokens[0]))

    return tokenizer.decode(tokens[0])

    #Returns the final response instead of printing it


#next_token_id = sample_next_token(logits, temperature=0.8, top_k=40)
#print("Sampled token:", tokenizer.decode(next_token_id[0]))



    40 → 'I'
   561 → ' would'
   588 → ' like'
   284 → ' to'
  2193 → ' learn'
   546 → ' about'
  2429 → ' gen'
 20185 → 'AI'
    11 → ','
   290 → ' and'
   314 → ' I'
   635 → ' also'
   588 → ' like'
  2438 → ' code'
    13 → '.'
  1867 → ' What'
   815 → ' should'
   314 → ' I'
   466 → ' do'
    30 → '?'


## Running the prompt with different temperatures

In [10]:
#Running the distillGPT2 prompt at 0.1 temperature and top-k=40
temp_01_distill = generate_step_by_step(text, steps=20, temp_var=0.1, top_k_var = 40)

print(temp_01_distill)

I would like to learn about genAI, and I also like code. What should I do?

I would like to learn about genAI, and I also like code. What should I do?


I would like to learn about genAI, and I also like code. What should I do?



I would like to learn about genAI, and I also like code. What should I do?




I would like to learn about genAI, and I also like code. What should I do?





I would like to learn about genAI, and I also like code. What should I do?






I would like to learn about genAI, and I also like code. What should I do?







I would like to learn about genAI, and I also like code. What should I do?








I would like to learn about genAI, and I also like code. What should I do?









I would like to learn about genAI, and I also like code. What should I do?










I would like to learn about genAI, and I also like code. What should I do?











I would like to learn about genAI, and I also like code. What should I do?












I would like to learn 

When temperature is set to 0.1, the model appeared to produce new line characters and blanks.

In [11]:
#Running the distillGPT2 prompt at 0.8 temperature and top-k=40
temp_08_distill = generate_step_by_step(text, steps=20, temp_var=0.8, top_k_var = 40)

print(temp_08_distill)


I would like to learn about genAI, and I also like code. What should I do?

I would like to learn about genAI, and I also like code. What should I do?


I would like to learn about genAI, and I also like code. What should I do?

The
I would like to learn about genAI, and I also like code. What should I do?

The code
I would like to learn about genAI, and I also like code. What should I do?

The code below
I would like to learn about genAI, and I also like code. What should I do?

The code below assumes
I would like to learn about genAI, and I also like code. What should I do?

The code below assumes that
I would like to learn about genAI, and I also like code. What should I do?

The code below assumes that what
I would like to learn about genAI, and I also like code. What should I do?

The code below assumes that what I
I would like to learn about genAI, and I also like code. What should I do?

The code below assumes that what I write
I would like to learn about genAI, and I also like 

When temperature is set to 0.8, the model produced a response relating to software and code. However, the response did not seem to address the question in the prompt.

In [12]:
#Running the distillGPT2 prompt at 2 temperature and top-k=40
temp_2_distill = generate_step_by_step(text, steps=20, temp_var=2, top_k_var = 40)

print(temp_2_distill)

I would like to learn about genAI, and I also like code. What should I do? Do
I would like to learn about genAI, and I also like code. What should I do? Do I
I would like to learn about genAI, and I also like code. What should I do? Do I need
I would like to learn about genAI, and I also like code. What should I do? Do I need a
I would like to learn about genAI, and I also like code. What should I do? Do I need a game
I would like to learn about genAI, and I also like code. What should I do? Do I need a game so
I would like to learn about genAI, and I also like code. What should I do? Do I need a game so fun
I would like to learn about genAI, and I also like code. What should I do? Do I need a game so fun at
I would like to learn about genAI, and I also like code. What should I do? Do I need a game so fun at 3
I would like to learn about genAI, and I also like code. What should I do? Do I need a game so fun at 3,
I would like to learn about genAI, and I also like code. What should I do

When temperature is set to 2.0, the model produced a response that did not seem to make sense, either as a response to the prompt or as a fragment.

# Distill GPT2 - Conclusions

I used the following [tutorial](https://www.geeksforgeeks.org/html/markdown-tables/) for making markdown tables as well as this [reference sheet](https://www.markdownlang.com/cheatsheet/).

The responses are graded with the following rubric:

| Description | Score Range |
|---| --- |
|(Perfect) Flawless grammar and logical flow.| 9-10|
|(High): Stays on topic; perhaps one slightly odd word choice. | 7–8 |
|(Fair): You can follow the "vibe," but there are logical leaps or awkward phrasing. | 5–6 |
|(Poor): Fragments of meaning exist, but the sentences don't connect. | 3–4
| (Incoherent): Total gibberish, random symbols, or word salad. | 1–2



The following are the results for the experiment above including my scores.


| Trial | Temperature (T) |	Predicted Behavior |	Model Response |	Model Coherence (1-10) | |
|---| --- | --- | --- | --- | ---|
|A	| 0.1 |	Conservative|  | 1
|B	|0.8	|Creative	|The code below assumes that what I write is not the code that I write in, but| 5
|C	|2.0	|Chaos | Do I need a game so fun at 3, 2 and perhaps as a kid if I can understand|	2

**Did your model repeat any words or phrases?**

The chaos model appeared to repeat phrases in that it produced new-line characters and blanks. The creative run repeated the phrase, "I write," but the creative and chaos models did not produce many repeating phrases. The creative and chaos models produced responses in the first person.

**Did the model use real words, or did it start outputting random characters and punctuation? Explain how the "Probability Distribution" changed to allow this.**

The chaos model used numbers separated by a comma. This did not make sense as a response to the prompt or as a fragment. I imagine that this is due to the lower number of layers for distillgpt. Fewer layers could mean that tokens receive logits that are less varied and more similar to each other, resulting in a probability distribution that has a reduced range. As a result, and despite the top-k setting, unrelated tokens could still rise to the top of the model's sampling.

**If you were building a medical AI to give prescriptions or advice, which temperature would you use?**

For distillgpt, I do not think I would use a temperature of 0.2, because this conservative model did not produce any response whatsoever. I would probably use a temperature of 0.8 or higher, because this was the most coherent response for the distilgpt2 model. I would, however, avoid using distilgpt2 for this kind of AI. There seems to be a greater likelihood of random tokens appearing in the response when one increases temperature.

**If you were building an AI to write a surrealist dream-journal, which would you use?**

For this model, I would probably use a temperature of 2.0, because the chaos model here produced a narrative response. It is possible that a temperature between 1.0 and 2.0 would be ideal for a surrealist dream-journal using distilgpt2, as the chaos model here produced a more random response, rather than surrealist.